# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



### Run SQl using DuckDB

In [1]:
# Download the Chinook sqlite database
%%bash
[ -f chinook.zip ] || curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -o chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [2]:
#Remove any old copy, then unzip
!rm -f chinook.db
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [3]:
# Check the files
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 15 18:30 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 12 16:40 sample_data


In [4]:
# Import Python libraries

import duckdb
import pandas as pd
import sqlite3 as db
import contextlib



con = duckdb.connect()                                  # an empty DuckDB session in memory
con.sql("INSTALL sqlite; LOAD sqlite;")                 # add-on that lets DuckDB read SQLite files
con.sql("ATTACH 'chinook.db' AS chinook (TYPE sqlite)") # plug in your file and call it 'chinook'
con.sql("USE chinook")                                  # make it the default, so you can write 'albums'

In [ ]:
# check: should list 11 tables + 2 sqlite_ tables
con.sql("SHOW TABLES").df()

,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoice_items
6,invoices
7,media_types
8,playlist_track
9,playlists


## How many columns

In [ ]:
#====================================================
# HOW MANY COLUMN IN EACH TABLE ?
#(note show what change if have diffrent database)
#====================================================

con.sql("""
    select table_name,                          -- never changes
           count(*) as n_columns                -- never changes (the name n_columns is your choice)
    from information_schema.columns             -- never changes
    where table_catalog = 'chinook'             -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'        -- keep only for SQLite files
    group by table_name                         -- never changes
    order by table_name                         -- never changes
""").df()

,table_name,n_columns
0,albums,3
1,artists,2
2,customers,13
3,employees,15
4,genres,2
5,invoice_items,5
6,invoices,9
7,media_types,2
8,playlist_track,2
9,playlists,2


## How many rows

In [ ]:
# ============================================================
# HOW MANY ROWS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Idea: ask "how many rows?" once per table, then stack all
#       the answers into one result table.
#
# TO REUSE ON ANOTHER DATABASE:
#   1. Run con.sql("SHOW TABLES").df() to get the table names.
#   2. Make one select line per table.
#   3. On each line, change the name in TWO places:
#        'label'      -> text shown in the output (in quotes)
#        from table   -> the real table being counted (no quotes)
#   4. Put "union all" BETWEEN lines (not before the first one,
#      not after the last one).
# ============================================================

con.sql("""
    -- Each select line answers: "How many rows are in this table?"
    --   'albums'           = a text label we type, so we know which table the number belongs to
    --   as table_name      = names the label column
    --   count(*)           = counts every row, even rows with blank values
    --   as n_rows          = names the count column
    --   from albums        = the table being counted (must match the label)

    select 'albums'         as table_name, count(*) as n_rows from albums
    union all                                   -- "stack the next answer under this one"
    select 'artists'        as table_name, count(*) as n_rows from artists
    union all
    select 'customers'      as table_name, count(*) as n_rows from customers
    union all
    select 'employees'      as table_name, count(*) as n_rows from employees
    union all
    select 'genres'         as table_name, count(*) as n_rows from genres
    union all
    select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items
    union all
    select 'invoices'       as table_name, count(*) as n_rows from invoices
    union all
    select 'media_types'    as table_name, count(*) as n_rows from media_types
    union all
    select 'playlist_track' as table_name, count(*) as n_rows from playlist_track
    union all
    select 'playlists'      as table_name, count(*) as n_rows from playlists
    union all
    select 'tracks'         as table_name, count(*) as n_rows from tracks

    order by table_name                         -- sort the final stacked table A to Z (written once, at the end)
""").df()                                       # .df() only DISPLAYS the result as a pandas table; SQL did the counting

# EASY MISTAKE: changing the 'label' but not the "from" table.
#   You'd get the wrong number under the right name, and NO error.

,table_name,n_rows
0,albums,347
1,artists,275
2,customers,59
3,employees,8
4,genres,25
5,invoice_items,2240
6,invoices,412
7,media_types,5
8,playlist_track,8715
9,playlists,18


## Breaking down the columns

### Identifier Columns

##### Find primary keys

In [ ]:
# ============================================================
# WHICH COLUMNS ARE PRIMARY KEYS?
# ------------------------------------------------------------
# duckdb_constraints() is DuckDB's built-in list of the "rules"
# on each table: primary keys, NOT NULL, and so on.
# We keep only the PRIMARY KEY rules.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           unnest(constraint_column_names) as column_name   -- unnest = split a list into one row per item
    from duckdb_constraints()                               -- built-in list of table rules
    where database_name = 'chinook'                         -- only our database
      and constraint_type = 'PRIMARY KEY'                   -- only primary-key rules
    order by table_name
""").df()

,table_name,column_name
0,albums,AlbumId
1,artists,ArtistId
2,customers,CustomerId
3,employees,EmployeeId
4,genres,GenreId
5,invoice_items,InvoiceLineId
6,invoices,InvoiceId
7,media_types,MediaTypeId
8,playlist_track,PlaylistId
9,playlist_track,TrackId


##### Find Foreign keys

In [ ]:
# ============================================================
# WHICH COLUMNS ARE FOREIGN KEYS (point to another table)?
# ------------------------------------------------------------
# DuckDB does NOT bring SQLite's foreign keys over when it
# attaches the file. So we use sqlite_query() to send a
# question straight to SQLite and get its answer back.
#
# Everything inside the single quotes '...' is SQLite's SQL.
# Inside that, '' (two single quotes) = one quote mark.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
#           (Only works for SQLite files.)
# ============================================================

con.sql("""
    select table_name, column_name
    from sqlite_query('chinook', '
        select m.name   as table_name,                  -- the table
               f."from" as column_name                  -- the column that points elsewhere
        from sqlite_master as m                          -- SQLite''s list of everything in the file
        join pragma_foreign_key_list(m.name) as f        -- foreign keys for each table
        where m.type = ''table''                         -- tables only (not indexes)
    ')
    order by table_name
""").df()

,table_name,column_name
0,albums,ArtistId
1,customers,SupportRepId
2,employees,ReportsTo
3,invoice_items,TrackId
4,invoice_items,InvoiceId
5,invoices,CustomerId
6,playlist_track,TrackId
7,playlist_track,PlaylistId
8,tracks,MediaTypeId
9,tracks,GenreId


##### Find how many PK,Fk

In [ ]:
# ============================================================
# HOW MANY PRIMARY KEYS, FOREIGN KEYS, AND IDENTIFIER COLUMNS
# ARE IN EACH TABLE?
# ------------------------------------------------------------
# "with" builds temporary named tables that only exist while
# this query runs. Here we build three, step by step:
#   1. primary_keys -> every primary-key column, tagged 'PK'
#   2. foreign_keys -> every foreign-key column, tagged 'FK'
#   3. all_keys     -> both lists stacked (UNION ALL keeps every row,
#                      so we can count PKs and FKs separately)
# Then the final select counts each kind per table.
#
# TO REUSE: change 'chinook' in BOTH places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: list every primary-key column, and tag it 'PK'
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name,  -- split key lists into one row per column
               'PK' as key_type                                 -- a label we add, so we know what kind of key it is
        from duckdb_constraints()                               -- DuckDB's built-in list of table rules
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 2: list every foreign-key column (asked directly from SQLite), and tag it 'FK'
    foreign_keys as (
        select table_name, column_name, 'FK' as key_type
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 3: stack both lists, keeping every row
    all_keys as (
        select * from primary_keys
        union all
        select * from foreign_keys
    )

    -- Final answer: count each kind of key per table
    select table_name,
           count(*) filter (where key_type = 'PK') as n_pk,                  -- count only the PK rows
           count(*) filter (where key_type = 'FK') as n_fk,                  -- count only the FK rows
           count(distinct column_name)            as n_identifier_columns   -- count each column once, even if it's both
    from all_keys
    group by table_name
    order by table_name
""").df()

,table_name,n_pk,n_fk,n_identifier_columns
0,albums,1,1,2
1,artists,1,0,1
2,customers,1,1,2
3,employees,1,1,2
4,genres,1,0,1
5,invoice_items,1,2,3
6,invoices,1,1,2
7,media_types,1,0,1
8,playlist_track,2,2,2
9,playlists,1,0,1


### Target Columns
**0.** Chinook has no defined prediction target. A target depends on the
question being asked, so none is chosen in this pass.

### Feature Columns

In [ ]:
# ============================================================
# HOW MANY FEATURE COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Feature columns = all columns that are NOT identifier columns.
# (No target chosen yet. When you pick one, subtract 1 more.)
#
# "with" builds temporary named tables for this query only:
#   1. all_columns        -> every column in every table
#   2. primary_keys       -> every primary-key column
#   3. foreign_keys       -> every foreign-key column
#   4. identifier_columns -> PK + FK combined, repeats removed
# Then we match all_columns against identifier_columns.
# Whatever does NOT match is a feature column.
#
# TO REUSE: change 'chinook' in all THREE places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: every column in every table
    all_columns as (
        select table_name, column_name
        from information_schema.columns                 -- one row per column
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer: match every column against the identifier list
    select a.table_name,
           count(*)                        as n_columns,             -- all columns
           count(i.column_name)            as n_identifier_columns,  -- the ones that matched
           count(*) - count(i.column_name) as n_feature_columns      -- the ones that did NOT match
    from all_columns as a
    left join identifier_columns as i                   -- keep EVERY column, matched or not
           on  a.table_name  = i.table_name             -- same table...
           and a.column_name = i.column_name            -- ...and same column name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_columns,n_identifier_columns,n_feature_columns
0,albums,3,2,1
1,artists,2,1,1
2,customers,13,2,11
3,employees,15,2,13
4,genres,2,1,1
5,invoice_items,5,3,2
6,invoices,9,2,7
7,media_types,2,1,1
8,playlist_track,2,2,0
9,playlists,2,1,1


#### Rows with nulls in each (and categories in each)

In [ ]:
# ============================================================
# FOR EACH COLUMN IN ONE TABLE:
#   how many rows are NULL (blank)?
#   how many CATEGORIES (different values) does it have?
# ------------------------------------------------------------
# columns(*) = "do this to EVERY column" (a DuckDB shortcut,
#              so you don't type each column name)
# unpivot    = turn one wide row into one row per column
#              (easier to read: column name | count)
#
# TO REUSE: change the table name (2 places).
# ============================================================

con.sql("""
    with
    -- Step 1: nulls per column -> one wide row, then turned into a list
    null_counts as (
        unpivot (
            select count(*) - count(columns(*))          -- all rows minus non-blank rows = blank rows
            from tracks
        )
        on columns(*)                                    -- turn every column into its own row
        into name column_name value n_null_rows          -- name the two new columns
    ),

    -- Step 2: different values per column -> same idea
    category_counts as (
        unpivot (
            select count(distinct columns(*))            -- how many different values (blanks not counted)
            from tracks
        )
        on columns(*)
        into name column_name value n_categories
    )

    -- Final answer: put the two lists side by side
    select n.column_name,
           n.n_null_rows,
           c.n_categories
    from null_counts as n
    join category_counts as c
      on n.column_name = c.column_name                   -- match each column to itself
    order by n.n_null_rows desc, n.column_name           -- columns with blanks first
""").df()

,column_name,n_null_rows,n_categories
0,Composer,978,852
1,AlbumId,0,347
2,Bytes,0,3501
3,GenreId,0,25
4,MediaTypeId,0,5
5,Milliseconds,0,3080
6,Name,0,3257
7,TrackId,0,3503
8,UnitPrice,0,2


In [ ]:
# ============================================================
# NULLS AND CATEGORIES FOR EVERY COLUMN IN EVERY TABLE
# ------------------------------------------------------------
# 1. SQL gives us the list of table names.
# 2. Python fills the SAME query pattern in once per table.
# 3. "union all" stacks them into ONE SQL query, which DuckDB runs.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

# 1. Ask SQL for the table names
tables = con.sql("""
    select table_name
    from information_schema.tables
    where table_catalog = 'chinook'              -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'         -- keep only for SQLite files
    order by table_name
""").df()["table_name"]

# 2. The query pattern for ONE table. {t} is a blank that gets the table name.
pattern = """
    select '{t}' as table_name, n.column_name, n.n_null_rows, c.n_categories
    from (unpivot (select count(*) - count(columns(*)) from {t})
          on columns(*) into name column_name value n_null_rows) as n
    join (unpivot (select count(distinct columns(*)) from {t})
          on columns(*) into name column_name value n_categories) as c
      on n.column_name = c.column_name
"""

# 3. Fill in the blank for every table, stack with "union all", run once
parts = [pattern.format(t=t) for t in tables]
query = ("select * from (" + "union all".join(parts) + ")"
         + " where n_null_rows > 0"                                  # keep only columns that have blanks
         + " order by table_name, n_null_rows desc, column_name")
con.sql(query).df()

,table_name,column_name,n_null_rows,n_categories
0,customers,Company,49,10
1,customers,Fax,47,12
2,customers,State,29,25
3,customers,PostalCode,4,55
4,customers,Phone,1,58
5,employees,ReportsTo,1,3
6,invoices,BillingState,202,25
7,invoices,BillingPostalCode,28,55
8,tracks,Composer,978,852


#### Data type

In [ ]:
# ============================================================
# WHAT DATA TYPES ARE IN THIS DATABASE?
# ------------------------------------------------------------
# Run this first on ANY new database, so you know which type
# names to treat as "numbers" in the next query.
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select data_type,                               -- the kind of value: number, text, date...
           count(*) as n_columns                    -- how many columns have this type
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by data_type
    order by n_columns desc
""").df()

,data_type,n_columns
0,VARCHAR,34
1,BIGINT,24
2,DOUBLE,3
3,TIMESTAMP,3


#### Numerical Columns

In [ ]:
# ============================================================
# HOW MANY NUMERICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Numerical column = a FEATURE column (not an ID) whose
#                    data type is a number type.
#
# Same steps as the feature-column query, plus ONE new thing:
#   is_number -> a true/false flag we add in Step 1
#                (true if the data type is a number type)
#
# TO REUSE: change 'chinook' in all THREE places.
#           If your database has other number types, add them
#           to the is_number list (check with the data-type query).
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus a true/false flag: is it a number type?
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL') as is_number   -- true or false
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null  = no match in the ID list = a feature column
    --   a.is_number            = the column holds numbers
    select a.table_name,
           count(*) filter (where i.column_name is null)                 as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number) as n_numerical_columns,
           string_agg(a.column_name, ', ')                                         -- glue the names into one text line
                    filter (where i.column_name is null and a.is_number) as numerical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_feature_columns,n_numerical_columns,numerical_columns
0,albums,1,0,None
1,artists,1,0,None
2,customers,11,0,None
3,employees,13,0,None
4,genres,1,0,None
5,invoice_items,2,2,"Quantity, UnitPrice"
6,invoices,7,1,Total
7,media_types,1,0,None
8,playlist_track,0,0,None
9,playlists,1,0,None


##### How many Discrete and Continuous Columns

In [ ]:
# ============================================================
# HOW MANY DISCRETE AND CONTINUOUS COLUMNS? (first guess, by type)
# ------------------------------------------------------------
# Only numerical FEATURE columns (not IDs) are sorted here.
#   is_whole_number   -> BIGINT, INTEGER...  -> guess: DISCRETE
#   is_decimal_number -> DOUBLE, DECIMAL...  -> guess: CONTINUOUS
# The type is only a first guess. Query 2 checks it against the data.
#
# TO REUSE: change 'chinook' in all THREE places.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus two flags: whole number or decimal?
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT') as is_whole_number,
               data_type in ('FLOAT', 'DOUBLE', 'DECIMAL')               as is_decimal_number
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer: count and list each kind (feature columns only)
    select a.table_name,
           count(*) filter (where i.column_name is null and a.is_whole_number)   as n_discrete_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_whole_number)   as discrete_columns,
           count(*) filter (where i.column_name is null and a.is_decimal_number) as n_continuous_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_decimal_number) as continuous_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_discrete_columns,discrete_columns,n_continuous_columns,continuous_columns
0,albums,0,None,0,None
1,artists,0,None,0,None
2,customers,0,None,0,None
3,employees,0,None,0,None
4,genres,0,None,0,None
5,invoice_items,1,Quantity,1,UnitPrice
6,invoices,0,None,1,Total
7,media_types,0,None,0,None
8,playlist_track,0,None,0,None
9,playlists,0,None,0,None


##### Reality Check

In [ ]:
# ============================================================
# REALITY CHECK: HOW MANY DIFFERENT VALUES DOES EACH
#                NUMERICAL COLUMN ACTUALLY HAVE?
# ------------------------------------------------------------
#   count(*)                -> how many rows
#   count(distinct column)  -> how many DIFFERENT values
# Few different values       -> behaves DISCRETE
# Close to one per row       -> behaves CONTINUOUS
#
# TO REUSE: one line per numerical column. On each line,
#           change the table name (2 places) and column name (2 places).
# ============================================================

con.sql("""
    select 'tracks'        as table_name, 'Milliseconds' as column_name,
           count(*)        as n_rows,
           count(distinct Milliseconds) as n_distinct_values
    from tracks
    union all
    select 'tracks',        'Bytes',     count(*), count(distinct Bytes)     from tracks
    union all
    select 'tracks',        'UnitPrice', count(*), count(distinct UnitPrice) from tracks
    union all
    select 'invoice_items', 'UnitPrice', count(*), count(distinct UnitPrice) from invoice_items
    union all
    select 'invoice_items', 'Quantity',  count(*), count(distinct Quantity)  from invoice_items
    union all
    select 'invoices',      'Total',     count(*), count(distinct Total)     from invoices
""").df()

,table_name,column_name,n_rows,n_distinct_values
0,tracks,Milliseconds,3503,3080
1,tracks,Bytes,3503,3501
2,tracks,UnitPrice,3503,2
3,invoice_items,UnitPrice,2240,2
4,invoice_items,Quantity,2240,1
5,invoices,Total,412,23


**Discrete / continuous — what the data shows:**
- Milliseconds (3,080 values) and Bytes (3,501) behave **continuous**,
  even though they're stored as whole numbers.
- UnitPrice (both tables) has only 2 values, so it behaves **discrete**.
- Quantity is always 1, a constant.
- Total has 23 values: discrete-ish.

**Rows with nulls in each:** 0 for all six numerical columns.

#### Categorical Columns

In [ ]:
# ============================================================
# HOW MANY CATEGORICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Categorical column = a FEATURE column (not an ID) stored as text.
#
# Same steps as the numerical query, with two more flags:
#   is_number   -> true if the type is a number type
#   is_category -> true if the type is text (labels)
#   is_date     -> true if the type is a date/time
# Every feature column should land in exactly ONE of these.
# CHECK: numerical + categorical + date = feature columns.
#
# TO REUSE: change 'chinook' in all THREE places.
#           Run the data-type query first; add any new type
#           names to the right flag list.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus three true/false flags for its kind
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL')   as is_number,
               data_type in ('VARCHAR', 'TEXT', 'BOOLEAN')   as is_category,
               data_type in ('DATE', 'TIMESTAMP', 'TIME')    as is_date
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null = not an ID = a feature column
    select a.table_name,
           count(*) filter (where i.column_name is null)                   as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number)   as n_numerical_columns,
           count(*) filter (where i.column_name is null and a.is_category) as n_categorical_columns,
           count(*) filter (where i.column_name is null and a.is_date)     as n_date_columns,
           string_agg(a.column_name, ', ')                                           -- list the names so you can see them
                    filter (where i.column_name is null and a.is_category) as categorical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_feature_columns,n_numerical_columns,n_categorical_columns,n_date_columns,categorical_columns
0,albums,1,0,1,0,Title
1,artists,1,0,1,0,Name
2,customers,11,0,11,0,"Email, Company, State, Fax, LastName, City, Po..."
3,employees,13,0,11,2,"Phone, FirstName, Address, Country, Email, Tit..."
4,genres,1,0,1,0,Name
5,invoice_items,2,2,0,0,None
6,invoices,7,1,5,1,"BillingCity, BillingAddress, BillingCountry, B..."
7,media_types,1,0,1,0,Name
8,playlist_track,0,0,0,0,None
9,playlists,1,0,1,0,Name


#### Nominal / ordinal

In [ ]:
# ============================================================
# LOOK AT THE VALUES: do these labels have a natural order?
# ------------------------------------------------------------
# Lists each different value and how many rows have it.
# Read the list and ask: "Is one of these higher/lower than another?"
#   No  -> NOMINAL
#   Yes -> ORDINAL
#
# TO REUSE: change the column name (3 places) and the table name.
# ============================================================

con.sql("""
    select Title,                          -- the column we're judging
           count(*) as n_employees         -- how many rows have each value
    from employees
    group by Title                         -- one row per different title
    order by n_employees desc              -- most common first
""").df()

,Title,n_employees
0,Sales Support Agent,3
1,IT Staff,2
2,General Manager,1
3,Sales Manager,1
4,IT Manager,1


**Rows with nulls in each:** [list the columns from the filtered
nulls output, e.g. tracks.Composer 978, customers.Company ..., ...]

In [ ]:
# ============================================================
# HOW MANY NOMINAL AND ORDINAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Categorical column = feature column (not an ID) stored as text.
# Nominal vs ordinal can't be detected from the data type, so
# YOU decide, and list the ordinal ones in Step 5 by hand.
#   on the ordinal list  -> ORDINAL
#   not on the list      -> NOMINAL
#
# TO REUSE: change 'chinook' in all THREE places.
#           Edit the Step 5 list to match YOUR judgment.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus a flag: is it text (a category)?
    all_columns as (
        select table_name,
               column_name,
               data_type in ('VARCHAR', 'TEXT', 'BOOLEAN') as is_category
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    ),

    -- Step 5: YOUR judgment, a hand-made list of ordinal columns
    --   "values" builds a small table from rows you type.
    --   To add more: ('table', 'column'), ('table', 'column')
    ordinal_columns (table_name, column_name) as (
        values ('employees', 'Title')
    )

    -- Final answer
    --   i.column_name is null     = not an ID        (a feature)
    --   a.is_category             = stored as text   (categorical)
    --   o.column_name is not null = on YOUR ordinal list
    select a.table_name,
           count(*) filter (where i.column_name is null and a.is_category)                               as n_categorical_columns,
           count(*) filter (where i.column_name is null and a.is_category and o.column_name is null)     as n_nominal_columns,
           count(*) filter (where i.column_name is null and a.is_category and o.column_name is not null) as n_ordinal_columns,
           string_agg(a.column_name, ', ')
                    filter (where i.column_name is null and a.is_category and o.column_name is not null) as ordinal_columns
    from all_columns as a
    left join identifier_columns as i                 -- match against the ID list
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    left join ordinal_columns as o                    -- match against YOUR ordinal list
           on  a.table_name  = o.table_name
           and a.column_name = o.column_name
    group by a.table_name
    order by a.table_name
""").df()

,table_name,n_categorical_columns,n_nominal_columns,n_ordinal_columns,ordinal_columns
0,albums,1,1,0,None
1,artists,1,1,0,None
2,customers,11,11,0,None
3,employees,11,10,1,Title
4,genres,1,1,0,None
5,invoice_items,0,0,0,None
6,invoices,5,5,0,None
7,media_types,1,1,0,None
8,playlist_track,0,0,0,None
9,playlists,1,1,0,None


    **Ordinal:** 1 column, employees.Title. Job titles have a rank
    (managers above staff), recorded in ReportsTo.
    **Nominal:** every other text column.

    **Rows with nulls in each:** [list the columns from the filtered
    nulls output, e.g. tracks.Composer 978, customers.Company ..., ...]

    **Categories in each:** Title has 5. Columns like Name, Email, and
    Phone have almost one value per row, so they're unique labels,
    not true categories.

#### Nested Columns

**0 in every table**: SQLite has no nested data types

In [ ]:
# ============================================================
# HOW MANY NESTED COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Nested column = one cell holds several values.
# In DuckDB, nested types look like:
#   INTEGER[]  or  VARCHAR[]   -> a LIST    (ends in [])
#   STRUCT(...)                -> a record  (several named fields)
#   MAP(...)                   -> key/value pairs
#   JSON                       -> JSON text
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           count(*) filter (where data_type like '%[]'          -- lists end in []
                               or data_type like 'STRUCT%'      -- records start with STRUCT
                               or data_type like 'MAP%'         -- maps start with MAP
                               or data_type = 'JSON')           -- JSON
               as n_nested_columns
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by table_name
    order by table_name
""").df()

,table_name,n_nested_columns
0,albums,0
1,artists,0
2,customers,0
3,employees,0
4,genres,0
5,invoice_items,0
6,invoices,0
7,media_types,0
8,playlist_track,0
9,playlists,0


## Breaking down the rows

### Rows with nulls

In [ ]:
# ============================================================
# HOW MANY ROWS HAVE AT LEAST ONE NULL? (one table)
# ------------------------------------------------------------
# where columns(*) is not null -> keeps only rows where EVERY
#                                 column has a value (complete rows)
# rows with nulls = all rows - complete rows
#
# TO REUSE: change the table name (3 places).
# ============================================================

con.sql("""
    select count(*)                                                    as n_rows,
           (select count(*) from tracks where columns(*) is not null) as n_complete_rows,    -- every cell filled
           count(*) - (select count(*) from tracks where columns(*) is not null) as n_rows_with_nulls  -- at least one blank
    from tracks
""").df()

,n_rows,n_complete_rows,n_rows_with_nulls
0,3503,2525,978


In [ ]:
# ============================================================
# ROWS WITH NULLS AND DUPLICATE ROWS, EVERY TABLE
# ------------------------------------------------------------
# Duplicates here compare the whole row (including the ID),
# so expect 0 everywhere. See "Duplicate Rows" below for the "ignore the ID" check.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

# 1. Ask SQL for the table names
tables = con.sql("""
    select table_name
    from information_schema.tables
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    order by table_name
""").df()["table_name"]

# 2. The pattern for ONE table. {t} is the blank for the table name.
pattern = """
    select '{t}' as table_name,
            count(*)                                                           as n_rows,
            count(*) - (select count(*) from {t} where columns(*) is not null) as n_rows_with_nulls,
            count(*) - (select count(*) from (select distinct * from {t}))      as n_duplicate_rows
    from {t}
"""

# 3. Fill in every table, stack with "union all", run once
query = "union all".join(pattern.format(t=t) for t in tables) + "order by table_name"
con.sql(query).df()

,table_name,n_rows,n_rows_with_nulls,n_duplicate_rows
0,albums,347,0,0
1,artists,275,0,0
2,customers,59,50,0
3,employees,8,1,0
4,genres,25,0,0
5,invoice_items,2240,0,0
6,invoices,412,209,0
7,media_types,5,0,0
8,playlist_track,8715,0,0
9,playlists,18,0,0


### Duplicate Rows

In [ ]:
# ============================================================
# HOW MANY DUPLICATE ROWS? (one table, two ways)
# ------------------------------------------------------------
# distinct *                     -> one copy of each different row
# duplicates = all rows - distinct rows
#
# WAY A: compare ALL columns, including the ID
# WAY B: compare all columns EXCEPT the ID (more useful, see below)
#
# TO REUSE: change the table name (2 places in each)
#           and the ID column in WAY B.
# ============================================================

# WAY A: whole row, including the ID
con.sql("""
    select count(*)                                                     as n_rows,
           count(*) - (select count(*) from (select distinct * from tracks)) as n_duplicate_rows
    from tracks
""").df()

,n_rows,n_duplicate_rows
0,3503,0


In [ ]:
# WAY B: ignore the ID column ("exclude" = every column except this one)
con.sql("""
    select count(*)                                                                       as n_rows,
           count(*) - (select count(*) from (select distinct * exclude (TrackId) from tracks)) as n_duplicate_rows
    from tracks
""").df()

,n_rows,n_duplicate_rows
0,3503,0


### Missing rows

In [ ]:
# ============================================================
# MISSING ROWS, READING A: gaps in the ID sequence
# ------------------------------------------------------------
# expected rows = last ID - first ID + 1
# missing rows  = expected rows - actual rows
# (Assumes IDs are numbered 1, 2, 3... with no planned gaps.)
#
# TO REUSE: change the ID column (4 places) and the table name.
# ============================================================

con.sql("""
    select min(TrackId)                           as first_id,
           max(TrackId)                           as last_id,
           count(*)                               as n_rows,
           max(TrackId) - min(TrackId) + 1        as n_expected_rows,
           max(TrackId) - min(TrackId) + 1 - count(*) as n_missing_rows
    from tracks
""").df()

,first_id,last_id,n_rows,n_expected_rows,n_missing_rows
0,1,3503,3503,3503,0


In [ ]:
# ============================================================
# MISSING ROWS, READING B: foreign key points to nothing
# ------------------------------------------------------------
# left join keeps every track. If no album matches,
# the album side is blank (null): that album row is MISSING.
#
# TO REUSE: change both tables and the key column.
# ============================================================

con.sql("""
    select count(*) as n_tracks_with_missing_album
    from tracks as t
    left join albums as a
           on t.AlbumId = a.AlbumId
    where a.AlbumId is null              -- no matching album found
      and t.AlbumId is not null          -- (ignore tracks that simply have no album listed)
""").df()

,n_tracks_with_missing_album
0,0


### Rows: summary
- **Rows with nulls:** customers 50, employees 1, invoices 209, tracks 978 (all from Composer). Every other table: 0.
- **Duplicate rows:** 0 in every table; 0 duplicate tracks even ignoring TrackId.
- **Missing rows:** 0 gaps in TrackId (1–3503); 0 tracks pointing to a missing album.

## Phase 2. Map the Terrain

### The ERD: how the tables connect
11 foreign-key links. Almost all are one-to-many.
- **Chain (sales):** customers → invoices → invoice_items → tracks
- **Bridge (many-to-many):** playlist_track links playlists and tracks
- **Self-link:** employees.ReportsTo points to another employee (their manager)
- **Hub:** tracks connects the music, sales, and playlist sides

In [ ]:
# ============================================================
# THE MAP: WHICH COLUMN POINTS TO WHICH TABLE?
# ------------------------------------------------------------
# Each row is one link (one line on the ERD):
#   from_table.from_column  -->  to_table.to_column
# These are the columns you match on when you JOIN two tables.
#
# Same idea as the Pass 1 foreign-key query, plus two new columns:
#   "table" = the table it points to
#   "to"    = the column it points to (usually that table's ID)
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
#           (Only works for SQLite files.)
# ============================================================

con.sql("""
    select from_table, from_column, to_table, to_column
    from sqlite_query('chinook', '
        select m.name    as from_table,      -- the table that holds the link
               f."from"  as from_column,     -- the column that points elsewhere
               f."table" as to_table,        -- the table it points TO
               f."to"    as to_column        -- the column it points TO
        from sqlite_master as m
        join pragma_foreign_key_list(m.name) as f
        where m.type = ''table''
    ')
    order by from_table, from_column
""").df()

,from_table,from_column,to_table,to_column
0,albums,ArtistId,artists,ArtistId
1,customers,SupportRepId,employees,EmployeeId
2,employees,ReportsTo,employees,EmployeeId
3,invoice_items,InvoiceId,invoices,InvoiceId
4,invoice_items,TrackId,tracks,TrackId
5,invoices,CustomerId,customers,CustomerId
6,playlist_track,PlaylistId,playlists,PlaylistId
7,playlist_track,TrackId,tracks,TrackId
8,tracks,AlbumId,albums,AlbumId
9,tracks,GenreId,genres,GenreId


### What one row represents (the grain)

| Table | One row = one ... |
|---|---|
| artists | artist |
| albums | album (by one artist) |
| tracks | song |
| genres | genre label (Rock, Jazz...) |
| media_types | file format (MPEG, AAC...) |
| playlists | playlist |
| playlist_track | one song on one playlist (a pairing) |
| customers | customer |
| employees | employee |
| invoices | purchase / receipt (one customer, one date) |
| invoice_items | one song bought (a line on a receipt) |

**Why it matters:** "How many sales?" has two answers.
412 purchases (invoices) or 2,240 songs sold (invoice_items).
Always say which grain you're counting.

In [ ]:
# ============================================================
# PEEK AT A TABLE: what does one row look like?
# ------------------------------------------------------------
# select *  = every column
# limit 5   = only the first 5 rows (a peek, not the whole table)
# Read a row and finish the sentence: "one row = one ___"
#
# TO REUSE: change the table name. Run once per table.
# ============================================================

con.sql("""
    select *
    from invoices
    limit 5
""").df()

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2009-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2009-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2009-01-06,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11,69 Salem Street,Boston,MA,USA,2113,13.86


In [ ]:
# ============================================================
# SAME SALES, TWO GRAINS: receipts vs. lines on receipts
# ------------------------------------------------------------
# invoices:      one row = one purchase (a receipt)
#                -> Total is the receipt total
# invoice_items: one row = one track bought (a line on the receipt)
#                -> UnitPrice * Quantity is the line price
# If the grains are right, the two sums should MATCH,
# but the row counts should be DIFFERENT.
#
# round(..., 2) = round to 2 decimal places (dollars and cents)
# ============================================================

con.sql("""
    select 'invoices'      as table_name,
           count(*)        as n_rows,                              -- how many receipts
           round(sum(Total), 2) as total_sales                     -- add up receipt totals
    from invoices
    union all
    select 'invoice_items',
           count(*),                                               -- how many lines
           round(sum(UnitPrice * Quantity), 2)                     -- add up line prices
    from invoice_items
""").df()

,table_name,n_rows,total_sales
0,invoices,412,2328.6
1,invoice_items,2240,2328.6


**Check:** both tables total \$2,328.60, so the grains are right:
same money, two levels of detail.

### Joins

#### Join 1: Inner join (albums + artists)

In [ ]:
# ============================================================
# JOIN 1 (INNER JOIN): each album with its artist's name
# ------------------------------------------------------------
# albums  has: AlbumId, Title, ArtistId   (only the artist's NUMBER)
# artists has: ArtistId, Name             (the artist's NAME)
# The join matches rows where the ArtistId is the same in both.
#
#   from albums as al     -> "al" is a short nickname for albums
#   join artists as ar    -> "ar" is a short nickname for artists
#   on al.ArtistId = ar.ArtistId
#                         -> the matching rule (the ERD link)
#   al.Title / ar.Name    -> "nickname.column" says WHICH table
#                            each column comes from
#
# "join" on its own = INNER JOIN: keep only rows that found a match.
#
# TO REUSE: change the two tables, their nicknames, and the
#           matching columns in "on" (look them up in THE MAP).
# ============================================================

con.sql("""
    select al.AlbumId,
           al.Title as album_title,                -- from albums
           ar.Name  as artist_name                 -- from artists
    from albums as al
    join artists as ar
      on al.ArtistId = ar.ArtistId                 -- match on the shared ID
    order by al.AlbumId
    limit 10                                       -- peek at the first 10
""").df()

,AlbumId,album_title,artist_name
0,1,For Those About To Rock We Salute You,AC/DC
1,2,Balls to the Wall,Accept
2,3,Restless and Wild,Accept
3,4,Let There Be Rock,AC/DC
4,5,Big Ones,Aerosmith
5,6,Jagged Little Pill,Alanis Morissette
6,7,Facelift,Alice In Chains
7,8,Warner 25 Anos,Antônio Carlos Jobim
8,9,Plays Metallica By Four Cellos,Apocalyptica
9,10,Audioslave,Audioslave


In [ ]:
# ============================================================
# CHECK: same number of rows before and after the join?
# ------------------------------------------------------------
# (select count(*) from ...) = a small query inside a query,
# so we can show two counts side by side in one row.
# ============================================================

con.sql("""
    select (select count(*) from albums)                       as n_albums,
           (select count(*)
            from albums as al
            join artists as ar on al.ArtistId = ar.ArtistId)   as n_rows_after_join
""").df()

,n_albums,n_rows_after_join
0,347,347


**Result:** 347 albums before and after the join, so every album has an artist.
Inner join keeps only rows that find a match. Grain is unchanged: one row = one album.

#### Join 2: Left join (artists without albums)

In [ ]:
# ============================================================
# JOIN 2 (LEFT JOIN): artists who have NO albums
# ------------------------------------------------------------
# left join = keep EVERY artist (the "left" table, listed first),
#             even if no album matches.
#             No match -> the album columns come back blank (null).
#
# where al.AlbumId is null  -> keep only the artists whose
#                              album side is blank = no albums
#
# TO REUSE: put the table you want to KEEP ALL OF first,
#           then "left join" the table you're looking up.
# ============================================================

con.sql("""
    select ar.ArtistId,
           ar.Name  as artist_name,
           al.Title as album_title                 -- will be blank (None) for these artists
    from artists as ar                             -- LEFT table: keep every artist
    left join albums as al
           on ar.ArtistId = al.ArtistId            -- same matching rule as Join 1
    where al.AlbumId is null                       -- no album found
    order by ar.ArtistId
    limit 10                                       -- peek at the first 10
""").df()

,ArtistId,artist_name,album_title
0,25,Milton Nascimento & Bebeto,None
1,26,Azymuth,None
2,28,João Gilberto,None
3,29,Bebel Gilberto,None
4,30,Jorge Vercilo,None
5,31,Baby Consuelo,None
6,32,Ney Matogrosso,None
7,33,Luiz Melodia,None
8,34,Nando Reis,None
9,35,Pedro Luís & A Parede,None


In [ ]:
# ============================================================
# CHECK: how many rows does each kind of join give?
# ------------------------------------------------------------
# n_artists                -> all artists
# n_inner_join_rows        -> artist+album pairs that MATCHED
# n_left_join_rows         -> matched pairs + artists with no match
# n_artists_without_albums -> the difference (the blanks)
# CHECK: inner + without_albums = left
# ============================================================

con.sql("""
    select (select count(*) from artists)                              as n_artists,
           (select count(*)
            from artists as ar
            join albums as al on ar.ArtistId = al.ArtistId)            as n_inner_join_rows,
           (select count(*)
            from artists as ar
            left join albums as al on ar.ArtistId = al.ArtistId)       as n_left_join_rows,
           (select count(*)
            from artists as ar
            left join albums as al on ar.ArtistId = al.ArtistId
            where al.AlbumId is null)                                  as n_artists_without_albums
""").df()

,n_artists,n_inner_join_rows,n_left_join_rows,n_artists_without_albums
0,275,347,418,71


**Result:** 71 of 275 artists have no albums. An inner join would hide them;
the left join keeps them with blank album columns.
**Grain warning:** after joining, one row = one artist–album pair (418 rows), not one artist.

#### Join 3: Chain join (customer → what they bought)

In [ ]:
# ============================================================
# JOIN 3 (CHAIN JOIN): what did one customer buy?
# ------------------------------------------------------------
# Walk the sales chain from the ERD, one link per "join":
#   customers     --CustomerId-->  invoices
#   invoices      --InvoiceId -->  invoice_items
#   invoice_items --TrackId   -->  tracks
#
# c.FirstName || ' ' || c.LastName
#   -> || glues text together: "Luís" + " " + "Gonçalves"
# where c.CustomerId = 1  -> just one customer, so it's easy to read
#
# TO REUSE: add one "join ... on ..." line per link in your chain.
#           Each "on" uses the matching columns from THE MAP.
# ============================================================

con.sql("""
    select c.FirstName || ' ' || c.LastName as customer_name,   -- from customers
           i.InvoiceId,                                         -- from invoices
           i.InvoiceDate,
           t.Name       as track_name,                          -- from tracks
           ii.UnitPrice                                         -- from invoice_items
    from customers as c
    join invoices      as i  on c.CustomerId = i.CustomerId     -- link 1
    join invoice_items as ii on i.InvoiceId  = ii.InvoiceId     -- link 2
    join tracks        as t  on ii.TrackId   = t.TrackId        -- link 3
    where c.CustomerId = 1                                      -- one customer only
    order by i.InvoiceDate, t.Name
""").df()

,customer_name,InvoiceId,InvoiceDate,track_name,UnitPrice
0,Luís Gonçalves,98,2010-03-11,Experiment In Terra,1.99
1,Luís Gonçalves,98,2010-03-11,Take the Celestra,1.99
2,Luís Gonçalves,121,2010-06-13,Calling Dr. Love,0.99
3,Luís Gonçalves,121,2010-06-13,Cold Gin,0.99
4,Luís Gonçalves,121,2010-06-13,Shout It Out Loud,0.99
5,Luís Gonçalves,121,2010-06-13,Strutter,0.99
6,Luís Gonçalves,143,2010-09-15,Back off Bitch,0.99
7,Luís Gonçalves,143,2010-09-15,Coma,0.99
8,Luís Gonçalves,143,2010-09-15,Don't Cry (Original),0.99
9,Luís Gonçalves,143,2010-09-15,Garden of Eden,0.99


In [ ]:
# ============================================================
# CHECK: rows after the chain = rows in which table?
# ------------------------------------------------------------
# The chain ends at the most detailed table (invoice_items),
# so the result should have one row per LINE ON A RECEIPT.
# ============================================================

con.sql("""
    select (select count(*) from invoice_items)                         as n_invoice_items,
           (select count(*)
            from customers as c
            join invoices      as i  on c.CustomerId = i.CustomerId
            join invoice_items as ii on i.InvoiceId  = ii.InvoiceId
            join tracks        as t  on ii.TrackId   = t.TrackId)       as n_rows_after_chain
""").df()

,n_invoice_items,n_rows_after_chain
0,2240,2240


In [ ]:
# ============================================================
# THE TRAP: adding up a column from the WRONG grain
# ------------------------------------------------------------
# After joining invoices to invoice_items, each receipt's Total
# repeats on EVERY line of that receipt.
#   right_total -> add up line prices (the grain we're at)    ✓
#   wrong_total -> add up receipt Totals, repeated per line   ✗
# right_total should match the $2,328.60 from Step 2.
# ============================================================

con.sql("""
    select round(sum(ii.UnitPrice * ii.Quantity), 2) as right_total,   -- one price per line
           round(sum(i.Total), 2)                    as wrong_total    -- receipt total, counted once PER LINE
    from invoices as i
    join invoice_items as ii on i.InvoiceId = ii.InvoiceId
""").df()

,right_total,wrong_total
0,2328.6,20848.62


**Result:** the chain has 2,240 rows, the same as invoice_items.
A chain join takes the grain of its most detailed table: one row = one song bought.
**Trap:** after joining, invoices.Total repeats on every line. Summing it gives
\$20,848.62 instead of \$2,328.60. Always add up a column from the grain you're at.

#### Join 4: Bridge join (playlists ↔ tracks)

In [ ]:
# ============================================================
# JOIN 4 (BRIDGE JOIN): the songs on one playlist
# ------------------------------------------------------------
# playlists and tracks are MANY-TO-MANY, so they connect
# THROUGH the bridge table playlist_track:
#   playlists      --PlaylistId-->  playlist_track
#   playlist_track --TrackId   -->  tracks
# The bridge only holds ID pairs, so we join on both sides
# to turn the numbers into names.
#
# TO REUSE: change the playlist name in the "where" line.
# ============================================================

con.sql("""
    select p.Name as playlist_name,                              -- from playlists
           t.Name as track_name                                  -- from tracks
    from playlists as p
    join playlist_track as pt on p.PlaylistId = pt.PlaylistId    -- playlist -> pairings
    join tracks         as t  on pt.TrackId   = t.TrackId        -- pairings -> songs
    where p.Name = 'Grunge'                                      -- one small playlist
    order by t.Name
""").df()

,playlist_name,track_name
0,Grunge,Alive
1,Grunge,Black Hole Sun
2,Grunge,Come As You Are
3,Grunge,Daughter
4,Grunge,Drain You
5,Grunge,Evenflow
6,Grunge,Hunger Strike
7,Grunge,In Bloom
8,Grunge,Jeremy
9,Grunge,Lithium


In [ ]:
# ============================================================
# HOW MANY SONGS ARE ON EACH PLAYLIST?
# ------------------------------------------------------------
# left join  -> keep EVERY playlist, even ones with no songs
# group by   -> one pile per playlist
# count(pt.TrackId) -> counts pairings, and SKIPS blanks,
#                      so an empty playlist shows 0
#   (count(*) would count the blank row and wrongly show 1)
# ============================================================

con.sql("""
    select p.PlaylistId,
           p.Name            as playlist_name,
           count(pt.TrackId) as n_songs                          -- blanks not counted
    from playlists as p
    left join playlist_track as pt on p.PlaylistId = pt.PlaylistId
    group by p.PlaylistId, p.Name                                -- one pile per playlist
    order by n_songs desc
""").df()

,PlaylistId,playlist_name,n_songs
0,8,Music,3290
1,1,Music,3290
2,5,90’s Music,1477
3,10,TV Shows,213
4,3,TV Shows,213
5,12,Classical,75
6,11,Brazilian Music,39
7,17,Heavy Metal Classic,26
8,14,Classical 101 - Next Steps,25
9,13,Classical 101 - Deep Cuts,25


In [ ]:
# ============================================================
# WHICH SONGS ARE ON THE MOST PLAYLISTS?
# ------------------------------------------------------------
# Same bridge, read from the OTHER side:
# one pile per SONG, counting how many playlists it's on.
# ============================================================

con.sql("""
    select t.TrackId,
           t.Name   as track_name,
           count(*) as n_playlists                               -- one pairing = one playlist
    from tracks as t
    join playlist_track as pt on t.TrackId = pt.TrackId
    group by t.TrackId, t.Name
    order by n_playlists desc, t.TrackId
    limit 10
""").df()

,TrackId,track_name,n_playlists
0,3403,Intoitus: Adorate Deum,5
1,3404,"Miserere mei, Deus",5
2,3408,"Aria Mit 30 Veränderungen, BWV 988 ""Goldberg V...",5
3,3409,"Suite for Solo Cello No. 1 in G Major, BWV 100...",5
4,3410,"The Messiah: Behold, I Tell You a Mystery... T...",5
5,3411,Solomon HWV 67: The Arrival of the Queen of Sheba,5
6,3415,Symphony No.5 in C Minor: I. Allegro con brio,5
7,3416,Ave Maria,5
8,3417,"Nabucco: Chorus, ""Va, Pensiero, Sull'ali Dorate""",5
9,3418,Die Walküre: The Ride of the Valkyries,5


**Result:** playlists and tracks are many-to-many, linked through playlist_track.
- 18 playlists; some have 0 songs (shown by the left join), some share a name.
- Songs per playlist add up to 8,715 = rows in playlist_track.
- Some songs appear on several playlists, so 8,715 pairings > 3,503 songs.

**Note:** with a left join, use count(column), not count(*), so empty groups show 0.

**Quirk:** two pairs of playlists are duplicates: "Music" (IDs 1 and 8, 3,290 songs each)
and "TV Shows" (IDs 3 and 10, 213 each). Same name, same size, different IDs.

#### Join 5: Self-join (employees → their manager)

In [ ]:
# ============================================================
# JOIN 5 (SELF-JOIN): each employee and their manager
# ------------------------------------------------------------
# employees.ReportsTo = the EmployeeId of that person's manager,
# and the manager is ALSO in the employees table.
# So we use the same table TWICE, with two nicknames:
#   emp -> the employee
#   mgr -> the manager (looked up by ID)
#
# left join -> keep EVERY employee, even the one with no manager
#              (the top boss: ReportsTo is blank)
#
# TO REUSE: any table with a column that points to its own ID
#           (e.g. a "parent" or "referred_by" column).
# ============================================================

con.sql("""
    select emp.EmployeeId,
           emp.FirstName || ' ' || emp.LastName as employee_name,
           emp.Title                            as employee_title,
           mgr.FirstName || ' ' || mgr.LastName as manager_name,     -- blank for the top boss
           mgr.Title                            as manager_title
    from employees as emp                                            -- copy 1: the employee
    left join employees as mgr                                       -- copy 2: the manager
           on emp.ReportsTo = mgr.EmployeeId                         -- "my boss's ID = their ID"
    order by emp.EmployeeId
""").df()

,EmployeeId,employee_name,employee_title,manager_name,manager_title
0,1,Andrew Adams,General Manager,None,None
1,2,Nancy Edwards,Sales Manager,Andrew Adams,General Manager
2,3,Jane Peacock,Sales Support Agent,Nancy Edwards,Sales Manager
3,4,Margaret Park,Sales Support Agent,Nancy Edwards,Sales Manager
4,5,Steve Johnson,Sales Support Agent,Nancy Edwards,Sales Manager
5,6,Michael Mitchell,IT Manager,Andrew Adams,General Manager
6,7,Robert King,IT Staff,Michael Mitchell,IT Manager
7,8,Laura Callahan,IT Staff,Michael Mitchell,IT Manager


In [ ]:
# ============================================================
# HOW MANY DIRECT REPORTS DOES EACH MANAGER HAVE?
# ------------------------------------------------------------
# Same self-join, but grouped by the MANAGER.
# Inner join this time: only people who ARE someone's manager.
# ============================================================

con.sql("""
    select mgr.EmployeeId,
           mgr.FirstName || ' ' || mgr.LastName as manager_name,
           mgr.Title                            as manager_title,
           count(*)                             as n_direct_reports   -- one row per person who reports to them
    from employees as emp
    join employees as mgr on emp.ReportsTo = mgr.EmployeeId
    group by mgr.EmployeeId, mgr.FirstName, mgr.LastName, mgr.Title
    order by n_direct_reports desc
""").df()

,EmployeeId,manager_name,manager_title,n_direct_reports
0,2,Nancy Edwards,Sales Manager,3
1,1,Andrew Adams,General Manager,2
2,6,Michael Mitchell,IT Manager,2


**Result:** employees.ReportsTo points back into employees, so the table is joined
to itself using two nicknames (emp, mgr).
- 8 employees; the General Manager has no manager (the 1 null found in Pass 1).
- 3 managers with 7 direct reports in total.
- Confirms Title is ordinal: General Manager → Manager → Agent/Staff.

### Phase 2 summary
- **Count rows before and after every join.**
- **Joins change the grain.** Know what one row means after you join.
- **Add up columns from the grain you're at** (see the wrong_total trap).

## Phase 3. Answer Guided Questions
Questions from the LucasMcL Chinook list. I answered 12 that build from
simple filters to joins, grouping, and top-N rankings.

#### Q1. Customers not in the US
Provide a query showing Customers (just their full names, customer ID and country)
who are not in the US.

In [5]:
# ============================================================
# Q1: CUSTOMERS WHO ARE NOT IN THE US
# ------------------------------------------------------------
# where Country != 'USA'  -> keep rows where Country is NOT 'USA'
#                            (!= means "not equal"; <> also works)
# FirstName || ' ' || LastName -> glue first + space + last name
# 'USA' must match the data exactly (capitals count).
#
# TO REUSE: change the column and value in the "where" line.
# ============================================================

con.sql("""
    select CustomerId,
           FirstName || ' ' || LastName as full_name,
           Country
    from customers
    where Country != 'USA'                -- everyone outside the US
    order by Country, full_name           -- group countries together, A to Z
""").df()

,CustomerId,full_name,Country
0,56,Diego Gutiérrez,Argentina
1,55,Mark Taylor,Australia
2,7,Astrid Gruber,Austria
3,8,Daan Peeters,Belgium
4,11,Alexandre Rocha,Brazil
5,10,Eduardo Martins,Brazil
6,13,Fernanda Ramos,Brazil
7,1,Luís Gonçalves,Brazil
8,12,Roberto Almeida,Brazil
9,32,Aaron Mitchell,Canada


In [6]:
# ============================================================
# CHECK: US + not-US should equal ALL customers (59)
# Also: how many different countries are outside the US?
# ============================================================

con.sql("""
    select count(*) filter (where Country =  'USA')                    as n_usa,
           count(*) filter (where Country != 'USA')                    as n_not_usa,
           count(*)                                                    as n_all_customers,
           count(distinct Country) filter (where Country != 'USA')     as n_countries_outside_us
    from customers
""").df()

,n_usa,n_not_usa,n_all_customers,n_countries_outside_us
0,13,46,59,23


**Answer:** 46 of 59 customers are outside the US (13 are in the US).
They come from 23 different countries; Canada has the most (8).

#### Q2. Customers from Brazil
Provide a query only showing the Customers from Brazil.

In [7]:
# ============================================================
# Q2: CUSTOMERS FROM BRAZIL
# ------------------------------------------------------------
# select *               -> every column (the question doesn't
#                           limit which details to show)
# where Country = 'Brazil' -> keep ONLY Brazilian customers
#                            (= means "equal"; spelling and capitals
#                             must match the data exactly)
#
# TO REUSE: change the value in quotes to any country.
# ============================================================

con.sql("""
    select *
    from customers
    where Country = 'Brazil'              -- only Brazil
    order by CustomerId
""").df()

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,10,Eduardo,Martins,Woodstock Discos,"Rua Dr. Falcão Filho, 155",São Paulo,SP,Brazil,01007-010,+55 (11) 3033-5446,+55 (11) 3033-4564,eduardo@woodstock.com.br,4
2,11,Alexandre,Rocha,Banco do Brasil S.A.,"Av. Paulista, 2022",São Paulo,SP,Brazil,01310-200,+55 (11) 3055-3278,+55 (11) 3055-8131,alero@uol.com.br,5
3,12,Roberto,Almeida,Riotur,"Praça Pio X, 119",Rio de Janeiro,RJ,Brazil,20040-020,+55 (21) 2271-7000,+55 (21) 2271-7070,roberto.almeida@riotur.gov.br,3
4,13,Fernanda,Ramos,None,Qe 7 Bloco G,Brasília,DF,Brazil,71020-677,+55 (61) 3363-5547,+55 (61) 3363-7855,fernadaramos4@uol.com.br,4


**Answer:** 5 customers are from Brazil: Luís Gonçalves, Eduardo Martins,
Alexandre Rocha, Roberto Almeida, and Fernanda Ramos.

#### Q3. Invoices of customers from Brazil
Provide a query showing the Invoices of customers who are from Brazil.
The resultant table should show the customer's full name, Invoice ID,
Date of the invoice and billing country.

In [8]:
# ============================================================
# Q3: INVOICES OF CUSTOMERS FROM BRAZIL
# ------------------------------------------------------------
# customers has the NAME and the customer's Country
# invoices  has InvoiceId, InvoiceDate, BillingCountry
# Join them on CustomerId (ERD link: invoices.CustomerId -> customers)
#
# where c.Country = 'Brazil' -> filter on the CUSTOMER'S country,
#                               because the question asks about
#                               "customers who are from Brazil"
#
# Grain after the join: one row = one invoice (not one customer),
# so each customer appears once per purchase.
#
# TO REUSE: change the country in the "where" line.
# ============================================================

con.sql("""
    select c.FirstName || ' ' || c.LastName as full_name,     -- from customers
           i.InvoiceId,                                       -- from invoices
           i.InvoiceDate,
           i.BillingCountry
    from customers as c
    join invoices as i
      on c.CustomerId = i.CustomerId                          -- match each invoice to its customer
    where c.Country = 'Brazil'                                -- customers FROM Brazil
    order by full_name, i.InvoiceDate
""").df()

,full_name,InvoiceId,InvoiceDate,BillingCountry
0,Alexandre Rocha,57,2009-09-06,Brazil
1,Alexandre Rocha,68,2009-10-17,Brazil
2,Alexandre Rocha,123,2010-06-17,Brazil
3,Alexandre Rocha,252,2012-01-22,Brazil
4,Alexandre Rocha,275,2012-04-25,Brazil
5,Alexandre Rocha,297,2012-07-28,Brazil
6,Alexandre Rocha,349,2013-03-18,Brazil
7,Eduardo Martins,25,2009-04-09,Brazil
8,Eduardo Martins,154,2010-11-14,Brazil
9,Eduardo Martins,177,2011-02-16,Brazil


In [9]:
# ============================================================
# CHECK: "customer from Brazil" vs "invoice billed to Brazil"
# ------------------------------------------------------------
# These COULD differ (a Brazilian customer could bill somewhere
# else). If both counts match, it doesn't matter which we filter on.
# count(distinct ...) -> how many different customers are involved
# ============================================================

con.sql("""
    select count(*) filter (where c.Country        = 'Brazil')               as n_invoices_customer_brazil,
           count(*) filter (where i.BillingCountry = 'Brazil')               as n_invoices_billed_brazil,
           count(distinct c.CustomerId) filter (where c.Country = 'Brazil')  as n_brazil_customers
    from customers as c
    join invoices as i on c.CustomerId = i.CustomerId
""").df()

,n_invoices_customer_brazil,n_invoices_billed_brazil,n_brazil_customers
0,35,35,5


**Answer:** the 5 Brazilian customers made 35 purchases (invoices), 7 each.
Filtering on the customer's country or the billing country gives the same 35,
but they're different columns and could differ in other data.

#### Q4. Employees who are Sales Agents
Provide a query showing only the Employees who are Sales Agents.

In [10]:
# ============================================================
# Q4 TRAP: an exact match on the question's wording
# ------------------------------------------------------------
# The question says "Sales Agents", but the data says
# "Sales Support Agent". "=" needs an EXACT match, so this
# returns 0 rows, and NO error to warn you.
# ============================================================

con.sql("""
    select EmployeeId, FirstName, LastName, Title
    from employees
    where Title = 'Sales Agent'           -- exact match: finds nothing
""").df()

,EmployeeId,FirstName,LastName,Title


In [11]:
# ============================================================
# Q4: EMPLOYEES WHO ARE SALES AGENTS (pattern match)
# ------------------------------------------------------------
# LIKE matches a PATTERN instead of the exact text:
#   %  = "any characters, any length (even none)"
#   _  = "exactly one character"
# '%Agent%' -> any title that contains the word "Agent"
#              anywhere: "Sales Support Agent" matches
#
# TO REUSE: put the key word between the % signs.
# ============================================================

con.sql("""
    select EmployeeId,
           FirstName || ' ' || LastName as full_name,
           Title,
           Email
    from employees
    where Title like '%Agent%'            -- title CONTAINS "Agent"
    order by EmployeeId
""").df()

,EmployeeId,full_name,Title,Email
0,3,Jane Peacock,Sales Support Agent,jane@chinookcorp.com
1,4,Margaret Park,Sales Support Agent,margaret@chinookcorp.com
2,5,Steve Johnson,Sales Support Agent,steve@chinookcorp.com


**Answer:** 3 employees are Sales Agents: Jane Peacock, Margaret Park, and Steve Johnson.
Their actual title is "Sales Support Agent", so an exact match on "Sales Agent"
returns 0 rows. LIKE '%Agent%' finds them by pattern.

#### Q5. Unique billing countries
Provide a query showing a unique list of billing countries from the Invoice table.

In [12]:
# ============================================================
# Q5: UNIQUE LIST OF BILLING COUNTRIES
# ------------------------------------------------------------
# select distinct -> keep only ONE copy of each different value
#                    (412 invoices -> one row per country)
# order by        -> sort the list A to Z
#
# TO REUSE: change the column and table.
# ============================================================

con.sql("""
    select distinct BillingCountry
    from invoices
    order by BillingCountry               -- A to Z
""").df()

,BillingCountry
0,Argentina
1,Australia
2,Austria
3,Belgium
4,Brazil
5,Canada
6,Chile
7,Czech Republic
8,Denmark
9,Finland


In [13]:
# ============================================================
# CHECK: all rows vs. different values
# ------------------------------------------------------------
# count(*)                      -> every invoice (repeats included)
# count(distinct BillingCountry) -> each country counted once
#                                  (the same number as rows in Cell 2)
# ============================================================

con.sql("""
    select count(*)                       as n_invoices,
           count(distinct BillingCountry) as n_unique_countries
    from invoices
""").df()

,n_invoices,n_unique_countries
0,412,24


**Answer:** the 412 invoices were billed to 24 different countries.
This matches Q1: 23 countries outside the US + the US = 24.

#### Q9. Invoices and sales in 2009 and 2011
How many Invoices were there in 2009 and 2011?
What are the respective total sales for each of those years?

In [14]:
# ============================================================
# Q9 (part 1): ONE YEAR WITH BETWEEN
# ------------------------------------------------------------
# between A and B -> keep values from A to B, INCLUDING both ends
#                    '2009-01-01' to '2009-12-31' = all of 2009
# count(*)        -> how many invoices
# sum(Total)      -> total sales (invoices grain: one Total per receipt)
#
# CAUTION: '2009-12-31' means midnight at the START of Dec 31.
#   Chinook's times are all 00:00:00, so it works here. With real
#   times (e.g. 3pm on Dec 31) use:
#   InvoiceDate >= '2009-01-01' and InvoiceDate < '2010-01-01'
#
# TO REUSE: change both dates to another year.
# ============================================================

con.sql("""
    select count(*)             as n_invoices,
           round(sum(Total), 2) as total_sales
    from invoices
    where InvoiceDate between '2009-01-01' and '2009-12-31'
""").df()

,n_invoices,total_sales
0,83,449.46


In [15]:
# ============================================================
# Q9 (part 2): BOTH YEARS IN ONE QUERY
# ------------------------------------------------------------
# year(InvoiceDate)   -> pull just the year out of the date (2009)
# in (2009, 2011)     -> keep only those two years
#                        (shorter than "= 2009 or = 2011")
# group by invoice_year -> one pile per year, then count and sum each
#
# TO REUSE: change the years inside in ( ... ).
# ============================================================

con.sql("""
    select year(InvoiceDate)    as invoice_year,
           count(*)             as n_invoices,
           round(sum(Total), 2) as total_sales
    from invoices
    where year(InvoiceDate) in (2009, 2011)    -- only these two years
    group by invoice_year                      -- one row per year
    order by invoice_year
""").df()

,invoice_year,n_invoices,total_sales
0,2009,83,449.46
1,2011,83,469.58


**Answer:** 2009 had 83 invoices totaling \$449.46; 2011 had 83 invoices totaling \$469.58.
Same number of invoices, but 2011 sales were \$20.12 higher, so the average purchase was bigger.

#### Q11. Line items per invoice
Looking at the InvoiceLine table, provide a query that COUNTs the number of
line items for each Invoice.

In [16]:
# ============================================================
# Q11: HOW MANY LINE ITEMS ARE ON EACH INVOICE?
# ------------------------------------------------------------
# invoice_items: one row = one line on a receipt
# group by InvoiceId -> one pile per receipt
# count(*)           -> how many lines are in each pile
#
# Result: one row per invoice (back up to the invoice grain).
#
# TO REUSE: group by the ID you want one row per,
#           and count the rows in each group.
# ============================================================

con.sql("""
    select InvoiceId,
           count(*) as n_line_items           -- lines on this receipt
    from invoice_items
    group by InvoiceId                        -- one pile per invoice
    order by InvoiceId
""").df()

,InvoiceId,n_line_items
0,1,2
1,2,4
2,3,6
3,4,9
4,5,14
...,...,...
407,408,4
408,409,6
409,410,9
410,411,14


In [17]:
# ============================================================
# CHECK: did every line get counted exactly once?
# ------------------------------------------------------------
# n_invoices       -> number of piles (should be 412)
# n_lines_counted  -> all the pile counts added up (should be 2,240)
#
# The inner query (in parentheses) is Cell 2's result;
# the outer query adds it up.
# ============================================================

con.sql("""
    select count(*)          as n_invoices,
           sum(n_line_items) as n_lines_counted
    from (
        select InvoiceId, count(*) as n_line_items
        from invoice_items
        group by InvoiceId
    )
""").df()

,n_invoices,n_lines_counted
0,412,2240.0


In [18]:
# ============================================================
# WHAT SIZES DO INVOICES COME IN?
# ------------------------------------------------------------
# Group TWICE:
#   inner query -> lines per invoice (Cell 2)
#   outer query -> how many invoices have each size
# ============================================================

con.sql("""
    select n_line_items,
           count(*) as n_invoices                 -- invoices with this many lines
    from (
        select InvoiceId, count(*) as n_line_items
        from invoice_items
        group by InvoiceId
    )
    group by n_line_items
    order by n_line_items
""").df()

,n_line_items,n_invoices
0,1,59
1,2,117
2,4,59
3,6,59
4,9,59
5,14,59


**Answer:** all 412 invoices have line items, adding up to 2,240 lines.
Invoices come in only 6 sizes (1, 2, 4, 6, 9, or 14 lines). The most common is
2 lines (117 invoices); every other size has exactly 59, one per customer.
**Quirk:** that even pattern means Chinook's sales are generated sample data, not real shopping.
Q10 (invoice 37 only) = add `where InvoiceId = 37` to the Q11 query.

#### Q13. Line items with track name and artist
Provide a query that includes the purchased track name AND artist name
with each invoice line item.

In [19]:
# ============================================================
# Q13: EACH LINE ITEM WITH ITS TRACK NAME AND ARTIST NAME
# ------------------------------------------------------------
# Walk the chain from the ERD, one "join" per link:
#   invoice_items --TrackId -->  tracks    (gives the song name)
#   tracks        --AlbumId -->  albums    (the bridge to the artist)
#   albums        --ArtistId-->  artists   (gives the artist name)
# Tracks don't store the artist; albums do. So we go THROUGH albums.
#
# Grain: still one row = one line item (2,240 rows).
#
# TO REUSE: add one join per link; take each "on" from THE MAP.
# ============================================================

con.sql("""
    select ii.InvoiceLineId,
           ii.InvoiceId,
           t.Name   as track_name,                         -- from tracks
           ar.Name  as artist_name,                        -- from artists
           ii.UnitPrice,
           ii.Quantity
    from invoice_items as ii
    join tracks  as t  on ii.TrackId  = t.TrackId          -- link 1: line -> song
    join albums  as al on t.AlbumId   = al.AlbumId         -- link 2: song -> album
    join artists as ar on al.ArtistId = ar.ArtistId        -- link 3: album -> artist
    order by ii.InvoiceLineId
    limit 20                                               -- peek; the full result is 2,240 rows
""").df()

,InvoiceLineId,InvoiceId,track_name,artist_name,UnitPrice,Quantity
0,1,1,Balls to the Wall,Accept,0.99,1
1,2,1,Restless and Wild,Accept,0.99,1
2,3,2,Put The Finger On You,AC/DC,0.99,1
3,4,2,Inject The Venom,AC/DC,0.99,1
4,5,2,Evil Walks,AC/DC,0.99,1
5,6,2,Breaking The Rules,AC/DC,0.99,1
6,7,3,Dog Eat Dog,AC/DC,0.99,1
7,8,3,Overdose,AC/DC,0.99,1
8,9,3,Love In An Elevator,Aerosmith,0.99,1
9,10,3,Janie's Got A Gun,Aerosmith,0.99,1


In [20]:
# ============================================================
# CHECK: rows before vs. after the chain
# ------------------------------------------------------------
# If any track had no album, or any album no artist, the inner
# joins would silently DROP those lines. Same counts = nothing lost.
# ============================================================

con.sql("""
    select (select count(*) from invoice_items)                  as n_line_items,
           (select count(*)
            from invoice_items as ii
            join tracks  as t  on ii.TrackId  = t.TrackId
            join albums  as al on t.AlbumId   = al.AlbumId
            join artists as ar on al.ArtistId = ar.ArtistId)     as n_rows_after_chain
""").df()

,n_line_items,n_rows_after_chain
0,2240,2240


**Answer:** all 2,240 line items have a track and an artist (no rows lost in the
3-link chain). Tracks don't store the artist; the chain has to go through albums.
Q12 (track name only) is the first link of this chain.

#### Q18. Total sales by each sales agent
Provide a query that shows total sales made by each sales agent.

In [21]:
# ============================================================
# Q18: TOTAL SALES BY EACH SALES AGENT
# ------------------------------------------------------------
# Path: employees -> customers -> invoices
#   e.EmployeeId = c.SupportRepId -> DIFFERENT column names, SAME number
#                                    (from THE MAP: customers.SupportRepId
#                                     points to employees.EmployeeId)
#   c.CustomerId = i.CustomerId   -> each customer's invoices
#
# sum(i.Total) is SAFE here: we stop at invoices (one Total per receipt),
# with no join to invoice_items, so no Total gets repeated.
#
# group by -> one pile per agent
#
# TO REUSE: swap sum(i.Total) for count(*) to count invoices instead.
# ============================================================

con.sql("""
    select e.EmployeeId,
           e.FirstName || ' ' || e.LastName as agent_name,
           count(distinct c.CustomerId)     as n_customers,      -- customers assigned to this agent
           count(*)                         as n_invoices,       -- receipts from those customers
           round(sum(i.Total), 2)           as total_sales       -- money from those receipts
    from employees as e
    join customers as c on e.EmployeeId = c.SupportRepId         -- agent -> their customers
    join invoices  as i on c.CustomerId = i.CustomerId           -- customers -> their receipts
    group by e.EmployeeId, e.FirstName, e.LastName               -- one row per agent
    order by total_sales desc
""").df()

,EmployeeId,agent_name,n_customers,n_invoices,total_sales
0,3,Jane Peacock,21,146,833.04
1,4,Margaret Park,20,140,775.40
2,5,Steve Johnson,18,126,720.16


In [22]:
# ============================================================
# CHECK: agent totals added together = ALL sales ($2,328.60)?
# ------------------------------------------------------------
# If yes: every invoice belongs to exactly one agent, and none
#         were lost or double-counted by the joins.
# ============================================================

con.sql("""
    select round(sum(agent_sales), 2)                        as sum_of_agent_totals,
           (select round(sum(Total), 2) from invoices)       as all_sales
    from (
        select e.EmployeeId, sum(i.Total) as agent_sales
        from employees as e
        join customers as c on e.EmployeeId = c.SupportRepId
        join invoices  as i on c.CustomerId = i.CustomerId
        group by e.EmployeeId
    )
""").df()

,sum_of_agent_totals,all_sales
0,2328.6,2328.6


**Answer:** 3 sales agents handle all sales: Jane Peacock (\$833.04, 21 customers),
Margaret Park (\$775.40, 20), and Steve Johnson (\$720.16, 18).
Together they total \$2,328.60, which is every invoice.
Q22 (customers per agent) = the n_customers column.

#### Q21. Top sales agent overall
Which sales agent made the most in sales over all?

In [23]:
# ============================================================
# Q21: THE TOP SALES AGENT OVERALL
# ------------------------------------------------------------
# Same query as Q18, plus:
#   order by total_sales desc -> biggest first ("desc" = descending)
#   limit 1                   -> keep ONLY the first row = the top agent
#
# TO REUSE: limit 3 -> top 3; "asc" instead of "desc" -> the lowest.
# ============================================================

con.sql("""
    select e.FirstName || ' ' || e.LastName as agent_name,
           round(sum(i.Total), 2)           as total_sales
    from employees as e
    join customers as c on e.EmployeeId = c.SupportRepId
    join invoices  as i on c.CustomerId = i.CustomerId
    group by e.EmployeeId, e.FirstName, e.LastName
    order by total_sales desc                -- highest sales first
    limit 1                                  -- keep only the top agent
""").df()

,agent_name,total_sales
0,Jane Peacock,833.04


**Answer:** Jane Peacock made the most in sales overall: \$833.04, about 36% of all sales.
The gap to second place (\$57.64) is big, so LIMIT 1 isn't hiding a tie.

#### Q23. Total sales per country
Provide a query that shows the total sales per country.
Which country's customers spent the most?

In [24]:
# ============================================================
# Q23: TOTAL SALES PER COUNTRY
# ------------------------------------------------------------
# group by BillingCountry -> one pile per country
# sum(Total)              -> money from that country
# count(*)                -> invoices from that country (= Q14)
# count(distinct CustomerId) -> how many customers in that country
# sales_per_customer      -> total / customers: who spends the most
#                            PER PERSON, not just in total
#
# order by total_sales desc -> top country first
#
# TO REUSE: group by any column (city, state...) to total by that.
# ============================================================

con.sql("""
    select BillingCountry                                         as country,
           count(distinct CustomerId)                             as n_customers,
           count(*)                                               as n_invoices,
           round(sum(Total), 2)                                   as total_sales,
           round(sum(Total) / count(distinct CustomerId), 2)      as sales_per_customer
    from invoices
    group by BillingCountry                                       -- one row per country
    order by total_sales desc                                     -- top spender first
""").df()

,country,n_customers,n_invoices,total_sales,sales_per_customer
0,USA,13,91,523.06,40.24
1,Canada,8,56,303.96,37.99
2,France,5,35,195.10,39.02
3,Brazil,5,35,190.10,38.02
4,Germany,4,28,156.48,39.12
5,United Kingdom,3,21,112.86,37.62
6,Czech Republic,2,14,90.24,45.12
7,Portugal,2,14,77.24,38.62
8,India,2,13,75.26,37.63
9,Chile,1,7,46.62,46.62


In [25]:
# ============================================================
# CHECK: countries add up to all sales, and match Q5's count?
# ------------------------------------------------------------
# n_countries        -> should be 24 (same as Q5)
# sum_country_sales  -> should be $2,328.60 (all sales)
# ============================================================

con.sql("""
    select count(*)                  as n_countries,
           round(sum(country_sales), 2) as sum_country_sales
    from (
        select BillingCountry, sum(Total) as country_sales
        from invoices
        group by BillingCountry
    )
""").df()

,n_countries,sum_country_sales
0,24,2328.6


**Answer:** USA customers spent the most in total: \$523.06 (about 22% of all sales),
followed by Canada (\$303.96). But the USA also has the most customers (13).
**Per customer**, the Czech Republic leads (\$45.12 per customer vs. \$40.24 for the USA).
"Spent the most" depends on whether you mean in total or per person.
Q14 (invoices per country) = the n_invoices column.

 #### Q25. Top 5 most purchased tracks
Provide a query that shows the top 5 most purchased tracks over all.

In [26]:
# ============================================================
# Q25: TOP 5 MOST PURCHASED TRACKS
# ------------------------------------------------------------
# invoice_items: one row = one track bought
# join tracks           -> get the track NAME
# group by TrackId      -> one pile per track
#                          (group by the ID, not just the name:
#                           different songs can share a name)
# sum(Quantity)         -> how many copies sold
# order by ... desc     -> most purchased first
# limit 5               -> keep the top 5
#
# TO REUSE: change "limit 5" to any top-N.
# ============================================================

con.sql("""
    select t.TrackId,
           t.Name             as track_name,
           sum(ii.Quantity)   as times_purchased
    from invoice_items as ii
    join tracks as t on ii.TrackId = t.TrackId
    group by t.TrackId, t.Name                     -- one row per track
    order by times_purchased desc, t.TrackId       -- most bought first; ties by ID
    limit 5
""").df()

,TrackId,track_name,times_purchased
0,2,Balls to the Wall,2.0
1,8,Inject The Venom,2.0
2,9,Snowballed,2.0
3,20,Overdose,2.0
4,32,Deuces Are Wild,2.0


In [27]:
# ============================================================
# CHECK: how many tracks share each purchase count?
# ------------------------------------------------------------
# Inner query: purchases per track (Cell 2 without the limit)
# Outer query: how many tracks bought once, twice, ...
# If MANY tracks tie at the top count, "top 5" is just 5 of them,
# picked by the tiebreaker (TrackId), not truly the best 5.
# ============================================================

con.sql("""
    select times_purchased,
           count(*) as n_tracks                          -- tracks with this many purchases
    from (
        select TrackId, sum(Quantity) as times_purchased
        from invoice_items
        group by TrackId
    )
    group by times_purchased
    order by times_purchased desc
""").df()

,times_purchased,n_tracks
0,2.0,256
1,1.0,1728


**Answer:** no track was purchased more than 2 times, and 256 tracks are tied at 2.
So there is no true "top 5": LIMIT 5 just shows the first 5 tied tracks by TrackId.
1,728 tracks were bought once, so 1,984 of 3,503 tracks sold at least once
and 1,519 were never purchased.